# TP1 — Concepts clés du TALN : morphologie, syntaxe, sémantique et pragmatique
### EMSI — Traitement Automatique du Langage Naturel (TALN)
**Version étudiant**

**Place dans le cours :** Chapitre 1 — Introduction au TALN : concepts de base, niveaux d'analyse linguistique  
**Bibliothèques :** spaCy (fr_core_news_sm), pandas

**Objectifs**
- Distinguer les niveaux d'analyse linguistique (morphologie, syntaxe, sémantique, pragmatique) sur des exemples concrets.
- Extraire les traits morphologiques d'une phrase et générer des formes fléchies et dérivées.
- Analyser la structure syntaxique : catégories grammaticales, ambiguïté catégorielle et relations de dépendance.

## Mode d'emploi
1. **Renseignez votre identification** dans la cellule ci-dessous (nom, prénom, classe, groupe, site EMSI).
2. Exécutez les cellules **dans l'ordre** (`Maj + Entrée`). Les variables d'un exercice sont réutilisées dans les suivants.
3. Complétez les cellules marquées `# TODO` / `...  # ← À COMPLÉTER`. Ne modifiez pas les cellules « ✅ Vérification » : elles vous indiquent si votre code est correct.
4. Répondez aux questions **📝 Interprétation** directement dans la cellule de texte prévue (double-clic pour éditer).
5. Enregistrez le notebook sous le nom affiché par la cellule d'identification, puis déposez-le sur la plateforme.

In [ ]:
NUM_TP = 1
# 🪪 IDENTIFICATION — À REMPLIR AVANT DE COMMENCER
NOM = "elhannak"        
PRENOM = "badr"    
CLASSE = "Data Ai"    
GROUPE = "G2"     
SITE = "Maarif"       

SITES_EMSI = ["Roudani", "Maarif", "Centre", "CFC", "Oranger", "Moulay Youssef"]
assert NOM and PRENOM and CLASSE and GROUPE, "Complétez NOM, PRENOM, CLASSE et GROUPE."
assert SITE in SITES_EMSI, f"SITE doit être l'un de {SITES_EMSI}"
NOM_FICHIER = f"TP{NUM_TP}_{NOM.upper().replace(' ', '-')}_{PRENOM.capitalize()}_{CLASSE}_{GROUPE}_{SITE.replace(' ', '')}.ipynb"
print(f"Étudiant(e) : {PRENOM} {NOM.upper()} | Classe : {CLASSE} | Groupe : {GROUPE} | EMSI {SITE}")
print("➡️  Nom du fichier à rendre :", NOM_FICHIER)

Étudiant(e) : badr ELHANNAK | Classe : Data Ai | Groupe : G2 | EMSI Maarif
➡️  Nom du fichier à rendre : TP1_ELHANNAK_Badr_Data Ai_G2_Maarif.ipynb


## Installation et préparation de l'environnement
Exécutez la cellule d'installation (une seule fois par machine ; sur Google Colab, à chaque session). 

In [22]:
# Installation des bibliothèques (à exécuter une seule fois)
import subprocess
import sys
%pip install -q nltk spacy pandas scikit-learn scipy
for model in ("fr_core_news_sm", "en_core_web_sm"):
    subprocess.run([sys.executable, "-m", "spacy", "download", model, "-q"], check=True)
print("Installation terminée. Si un module reste introuvable : Noyau → Redémarrer, puis réexécuter.")

Note: you may need to restart the kernel to use updated packages.
Installation terminée. Si un module reste introuvable : Noyau → Redémarrer, puis réexécuter.


In [23]:
# Imports communs, ressources et outil d'auto-vérification
import warnings
warnings.filterwarnings("ignore")
import nltk
for ressource in ['punkt', 'punkt_tab', 'wordnet', 'omw-1.4', 'stopwords']:
    nltk.download(ressource, quiet=True)

def verifier(description, test):
    """Affiche ✅ si test() est vrai, ❌ sinon (sans interrompre le notebook)."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {description} — {type(e).__name__} : {e}")
        return
    print(("✅ " if ok else "❌ ") + description)
    return

print("Environnement prêt.")

Environnement prêt.


---
## Exercice 0 — Prise en main : les niveaux d'analyse sur une phrase

Un système de TALN analyse un énoncé à plusieurs **niveaux** : la **morphologie** (structure des mots), la **syntaxe** (organisation des mots en phrase), la **sémantique** (sens) et la **pragmatique** (sens en contexte, intention du locuteur). On découvre ici la chaîne de traitement (*pipeline*) de spaCy, qui produit en une seule passe des informations relevant de chacun de ces niveaux.

**Étape 1.** Exécutez la cellule suivante : elle charge le modèle français de spaCy et affiche les composants de son pipeline.

In [24]:
import spacy
nlp = spacy.load("fr_core_news_sm")
print("Composants du pipeline :", nlp.pipe_names)

phrase = "Les chercheuses de l'EMSI ont présenté hier leurs nouveaux résultats à Casablanca."
doc = nlp(phrase)
for t in doc:
    print(f"{t.text:<12} lemme={t.lemma_:<12} pos={t.pos_:<6} dep={t.dep_:<10} morph={t.morph}")

Composants du pipeline : ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Les          lemme=le           pos=DET    dep=det        morph=Definite=Def|Number=Plur|PronType=Art
chercheuses  lemme=chercheur    pos=NOUN   dep=nsubj      morph=Gender=Fem|Number=Plur
de           lemme=de           pos=ADP    dep=case       morph=
l'           lemme=le           pos=DET    dep=det        morph=Definite=Def|Number=Sing|PronType=Art
EMSI         lemme=emsi         pos=NOUN   dep=nmod       morph=Number=Sing
ont          lemme=avoir        pos=AUX    dep=aux:tense  morph=Mood=Ind|Number=Plur|Person=3|Tense=Pres|VerbForm=Fin
présenté     lemme=présenter    pos=VERB   dep=ROOT       morph=Gender=Masc|Number=Sing|Tense=Past|VerbForm=Part
hier         lemme=hier         pos=ADV    dep=advmod     morph=
leurs        lemme=leur         pos=DET    dep=det        morph=Number=Plur|Poss=Yes
nouveaux     lemme=nouveau      pos=ADJ    dep=amod       morph=Gender=Masc|Number=

**Étape 2.** Complétez le code : nombre de tokens de `doc`, liste des lemmes des verbes (`pos_ == "VERB"`) et liste des entités nommées `(texte, étiquette)`.

In [25]:
nb_tokens = len(doc)
lemmes_verbes = [t.lemma_ for t in doc if t.pos_ == "VERB"]
entites = [(e.text, e.label_) for e in doc.ents]
print(nb_tokens, lemmes_verbes, entites)

14 ['présenter'] [('EMSI', 'ORG'), ('Casablanca', 'LOC')]


In [26]:
# ✅ Vérification (ne pas modifier)
verifier("nombre de tokens correct", lambda: nb_tokens == len(doc))
verifier("lemmes des verbes correctement extraits", lambda: lemmes_verbes == [t.lemma_ for t in doc if t.pos_ == "VERB"])
verifier("entités nommées correctement extraites", lambda: entites == [(e.text, e.label_) for e in doc.ents])

✅ nombre de tokens correct
✅ lemmes des verbes correctement extraits
✅ entités nommées correctement extraites


**Étape 3.** Associez chaque observation au niveau d'analyse concerné : `"morphologie"`, `"syntaxe"`, `"sémantique"` ou `"pragmatique"`.

In [27]:
NIVEAUX = {
    "« chercheuses » = chercheu(r) + -euse + -s (féminin pluriel)": "morphologie",
    "« chercheuses » est le sujet du verbe « présenter »": "syntaxe",
    "« Casablanca » désigne une ville ; « résultats » ≈ « conclusions »": "sémantique",
    "« hier » renvoie à la veille du jour où la phrase est prononcée": "pragmatique",
    "« Tu peux fermer la fenêtre ? » est une demande, pas une question sur une capacité": "pragmatique",
    "« ont présenté » : auxiliaire avoir + participe passé (passé composé)": "morphologie",
}

In [28]:
# ✅ Vérification (ne pas modifier)
verifier("les 6 observations sont correctement classées", lambda: list(NIVEAUX.values()) ==
         ["morphologie", "syntaxe", "sémantique", "pragmatique", "pragmatique", "morphologie"])

✅ les 6 observations sont correctement classées


#### 📝 Interprétation 0.1
Quelle information du pipeline spaCy relève de chaque niveau (morphologie, syntaxe, sémantique) ? Quel niveau n'est **pas** couvert par spaCy et pourquoi est-il plus difficile à automatiser ?

**Votre réponse :**

La morphologie est représentée notamment par les lemmes (`lemma_`) et les traits flexionnels (`morph`), comme le genre, le nombre, la personne et le temps. La syntaxe est décrite par les catégories grammaticales (`pos_`) et les dépendances (`dep_`, `head`) qui relient les mots. La sémantique n'est que partiellement représentée : la reconnaissance d'entités nommées peut identifier « Casablanca » comme un lieu, mais le pipeline ne comprend pas à lui seul tout le sens de la phrase. La pragmatique n'est pas réellement couverte : interpréter « hier » ou reconnaître une demande indirecte exige de connaître la situation, le locuteur, l'interlocuteur et leurs intentions, informations souvent absentes du texte isolé.

---
## Exercice 1 — Morphologie : flexion et dérivation

La **morphologie** étudie la structure interne des mots. On distingue la **morphologie flexionnelle** (variation d'un même mot selon le genre, le nombre, le temps, la personne : *chercheur → chercheuses*, *parler → parlions*) et la **morphologie dérivationnelle** (création d'un nouveau mot par affixation : *organiser → désorganisation*).

### 1.1 Traits morphologiques d'une phrase
Construisez un DataFrame avec une ligne par token (hors ponctuation) et les colonnes `forme`, `lemme`, `pos` puis tous les traits de `t.morph.to_dict()` (astuce : `{**dico1, **dico2}` fusionne deux dictionnaires).

In [29]:
import pandas as pd
phrase = "Les chercheuses avaient présenté nos nouveaux résultats hier."
lignes = []
for t in nlp(phrase):
    if not t.is_punct:
        lignes.append({"forme": t.text, "lemme": t.lemma_, "pos": t.pos_, **t.morph.to_dict()})
traits = pd.DataFrame(lignes).fillna("")
traits

,forme,lemme,pos,Definite,Number,PronType,Gender,Mood,Person,Tense,VerbForm,Poss
0,Les,le,DET,Def,Plur,Art,,,,,,
1,chercheuses,chercheur,NOUN,,Plur,,Fem,,,,,
2,avaient,avoir,AUX,,Plur,,,Ind,3,Imp,Fin,
3,présenté,présenter,VERB,,Sing,,Masc,,,Past,Part,
4,nos,notre,DET,,Plur,,,,,,,Yes
5,nouveaux,nouveau,ADJ,,Plur,,Masc,,,,,
6,résultats,résultat,NOUN,,Plur,,Masc,,,,,
7,hier,hier,ADV,,,,,,,,,


In [30]:
# ✅ Vérification (ne pas modifier)
_mots = [t for t in nlp(phrase) if not t.is_punct]
_ch = next(t for t in _mots if t.text == "chercheuses")
verifier("une ligne par mot (8 mots, ponctuation exclue)", lambda: len(traits) == len(_mots) == 8)
verifier("colonnes forme, lemme et pos présentes", lambda: {"forme", "lemme", "pos"} <= set(traits.columns))
verifier("traits de « chercheuses » recopiés depuis token.morph",
         lambda: all(traits.set_index("forme").loc["chercheuses", k] == v for k, v in _ch.morph.to_dict().items()))

✅ une ligne par mot (8 mots, ponctuation exclue)
✅ colonnes forme, lemme et pos présentes
✅ traits de « chercheuses » recopiés depuis token.morph


#### 📝 Interprétation 1.1
Relevez dans le tableau un trait qui marque le **temps** et un trait qui marque l'**accord**. Pourquoi « avaient » et « présenté » portent-ils des traits différents alors qu'ils forment un seul temps verbal (plus-que-parfait) ?

**Votre réponse :**

Dans le tableau, `Tense=Imp` sur « avaient » marque l'imparfait, et `Number=Plur` avec `Person=3` indique l'accord de l'auxiliaire avec « les chercheuses ». « Chercheuses » porte aussi `Gender=Fem` et `Number=Plur`. Au plus-que-parfait, « avaient » est l'auxiliaire conjugué : il porte le temps, la personne et le nombre. « Présenté » est un participe passé, donc non fini (`VerbForm=Part`) ; il porte la valeur de participe passé (`Tense=Past`) mais pas la personne. Ici il reste au masculin singulier, car le complément d'objet « nos nouveaux résultats » suit le participe et ne déclenche pas l'accord avec celui-ci.

### 1.2 Générer la flexion verbale (1er groupe)
Écrivez `conjuguer(infinitif, temps)` qui renvoie les 6 formes (je, tu, il, nous, vous, ils) au présent (`"Pres"`), à l'imparfait (`"Imp"`) ou au futur (`"Fut"`).
- Radical : l'infinitif sans `-er` (présent, imparfait) ou l'infinitif entier (futur).
- Règle orthographique : devant une terminaison commençant par **a** ou **o**, un radical en *-g* prend un *e* (*mangeons*) et un radical en *-c* devient *-ç* (*commençons*). Complétez `adapter`.

In [31]:
TERMINAISONS = {
    "Pres": ("e", "es", "e", "ons", "ez", "ent"),
    "Imp":  ("ais", "ais", "ait", "ions", "iez", "aient"),
    "Fut":  ("ai", "as", "a", "ons", "ez", "ont"),
}

def adapter(radical, terminaison):
    """mangeons, commençons : ajustement orthographique devant a / o."""
    if terminaison.startswith(("a", "o")):
        if radical.endswith("g"):
            radical += "e"
        elif radical.endswith("c"):
            radical = radical[:-1] + "ç"
    return radical

def conjuguer(infinitif, temps):
    base = infinitif if temps == "Fut" else infinitif[:-2]
    return [adapter(base, t) + t for t in TERMINAISONS[temps]]

for v in ("parler", "manger", "commencer"):
    print(v, conjuguer(v, "Pres"), conjuguer(v, "Imp")[:2])

parler ['parle', 'parles', 'parle', 'parlons', 'parlez', 'parlent'] ['parlais', 'parlais']
manger ['mange', 'manges', 'mange', 'mangeons', 'mangez', 'mangent'] ['mangeais', 'mangeais']
commencer ['commence', 'commences', 'commence', 'commençons', 'commencez', 'commencent'] ['commençais', 'commençais']


In [32]:
# ✅ Vérification (ne pas modifier)
verifier("nous mangeons", lambda: conjuguer("manger", "Pres")[3] == "mangeons")
verifier("je commençais", lambda: conjuguer("commencer", "Imp")[0] == "commençais")
verifier("nous commencions (pas de ç devant i)", lambda: conjuguer("commencer", "Imp")[3] == "commencions")
verifier("ils parleront", lambda: conjuguer("parler", "Fut")[5] == "parleront")

✅ nous mangeons
✅ je commençais
✅ nous commencions (pas de ç devant i)
✅ ils parleront


**Validation par l'analyseur.** La cellule suivante (fournie) soumet chaque forme générée à spaCy et vérifie qu'il retrouve le bon lemme, le bon temps, la bonne personne et le bon nombre.

In [33]:
PRONOMS = ["je", "tu", "il", "nous", "vous", "ils"]
PERSONNES = ["1", "2", "3", "1", "2", "3"]
NOMBRES = ["Sing", "Sing", "Sing", "Plur", "Plur", "Plur"]
def chercherPartie(m,part):
    position=m.find(part)+len(part)
    p=position+1; ch=""
    while p<len(m) and m[p]!='|':
        p=p+1
    return m[position+1:p]
def valider(infinitif, temps):
    bons = 0
    for pron, forme, pers, nb in zip(PRONOMS, conjuguer(infinitif, temps), PERSONNES, NOMBRES):
        print((pron, forme, pers, nb))
        #print(pron.capitalize()+" "+forme)
        verbes = [t for t in nlp(pron.capitalize()+" "+forme) if t.pos_ in ("VERB", "AUX")]
        if verbes:
            m = verbes[0].morph
            mm=str(m); # print(mm)
            print((verbes[0].lemma_,infinitif), (chercherPartie(mm,"Tense"), [temps]), (chercherPartie(mm,"Person"),[pers]), (chercherPartie(mm,"Number"), [nb]))
            bons += (verbes[0].lemma_ == infinitif and m.get("Tense") == [temps]
                     and m.get("Person") == [pers] and m.get("Number") == [nb])
    return bons
    
resultats = pd.DataFrame({t: {v: f"{valider(v, t)}/6" for v in ("parler", "manger", "commencer", "voyager", "danser")}
                          for t in TERMINAISONS})
resultats

('je', 'parle', '1', 'Sing')
('parler', 'parler') ('Pres', ['Pres']) ('1', ['1']) ('Sing', ['Sing'])
('tu', 'parles', '2', 'Sing')
('parler', 'parler') ('Past', ['Pres']) ('3', ['2']) ('Sing', ['Sing'])
('il', 'parle', '3', 'Sing')
('nous', 'parlons', '1', 'Plur')
('parler', 'parler') ('Pres', ['Pres']) ('1', ['1']) ('Plur', ['Plur'])
('vous', 'parlez', '2', 'Plur')
('parler', 'parler') ('Pres', ['Pres']) ('2', ['2']) ('Plur', ['Plur'])
('ils', 'parlent', '3', 'Plur')
('parler', 'parler') ('Pres', ['Pres']) ('3', ['3']) ('Plur', ['Plur'])
('je', 'mange', '1', 'Sing')
('manger', 'manger') ('Pres', ['Pres']) ('1', ['1']) ('Sing', ['Sing'])
('tu', 'manges', '2', 'Sing')
('il', 'mange', '3', 'Sing')
('manger', 'manger') ('Pres', ['Pres']) ('3', ['3']) ('Sing', ['Sing'])
('nous', 'mangeons', '1', 'Plur')
('manger', 'manger') ('Pres', ['Pres']) ('1', ['1']) ('Plur', ['Plur'])
('vous', 'mangez', '2', 'Plur')
('manger', 'manger') ('Pres', ['Pres']) ('1', ['2']) ('Sing', ['Plur'])
('ils', 'mang

,Pres,Imp,Fut
parler,4/6,2/6,3/6
manger,3/6,1/6,3/6
commencer,3/6,1/6,2/6
voyager,3/6,2/6,3/6
danser,3/6,2/6,3/6


#### 📝 Interprétation 1.2
Les formes que vous générez sont correctes, pourtant l'analyseur ne les valide pas toutes. Donnez deux raisons possibles (pensez à l'ambiguïté des formes et à la taille du modèle `sm`).

**Votre réponse :**

Les formes peuvent être ambiguës : par exemple, « parle » peut correspondre à plusieurs personnes au présent, et certaines terminaisons sont homographes entre temps ou personnes. Le contexte aide, mais le petit modèle peut tout de même attribuer de mauvais traits : il analyse notamment « tu parles » comme une forme passée à la troisième personne et « nous commençons » comme un futur. De plus, `fr_core_news_sm` est un modèle statistique compact, entraîné sur des données limitées ; il connaît moins bien les formes et leurs contextes qu'un modèle plus grand ou spécialisé. Les formes générées peuvent donc être correctes même si l'analyseur ne les valide pas.

### 1.3 Morphologie dérivationnelle : segmenter préfixe + base + suffixe
Le segmenteur ci-dessous énumère toutes les façons de retirer des préfixes puis des suffixes. Complétez `base_valide(b)` : une base est valide si `b`, ou `b` + une terminaison parmi `"er", "ir", "re", "e", "r"`, appartient au `LEXIQUE`.

In [34]:
PREFIXES = ["anti", "contre", "inter", "sous", "sur", "pré", "dés", "dé", "ré", "re", "in", "im", "co"]
SUFFIXES = ["ement", "ation", "able", "ible", "eur", "euse", "ité", "ment"]
LEXIQUE = {"faire", "lire", "organiser", "utiliser", "chanter", "changer",
           "population", "construire", "voter", "possible", "créer"}


def base_valide(b):
    """Vrai si b, ou b + une terminaison, est un mot du LEXIQUE."""
    return any(b + terminaison in LEXIQUE for terminaison in ("", "er", "ir", "re", "e", "r"))


def enlever_prefixes(mot):
    """Liste de toutes les façons de retirer des préfixes : (reste, préfixes retirés)."""
    resultats = [(mot, [])]              # au départ : on n'a rien retiré
    niveau = [(mot, [])]                 # formes obtenues à l'étape précédente
    for etape in range(len(mot)):        # on ne peut pas retirer plus de préfixes que de lettres
        suivant = []
        for reste, prefs in niveau:
            for p in PREFIXES:
                if reste.startswith(p) and len(reste) > len(p) + 2:
                    suivant.append((reste[len(p):], prefs + [p]))
        resultats = resultats + suivant
        niveau = suivant                 # on recommence à partir des nouvelles formes
    return resultats


def enlever_suffixes(mot):
    """Liste de toutes les façons de retirer des suffixes : (base, suffixes retirés)."""
    resultats = [(mot, [])]
    niveau = [(mot, [])]
    for etape in range(len(mot)):
        suivant = []
        for reste, sufs in niveau:
            for s in SUFFIXES:
                if reste.endswith(s) and len(reste) > len(s) + 2:
                    suivant.append((reste[:-len(s)], [s] + sufs))
        resultats = resultats + suivant
        niveau = suivant
    return resultats


def segmenter(mot):
    meilleure = None
    meilleur_nb = 0
    for reste, prefs in enlever_prefixes(mot):
        for base, sufs in enlever_suffixes(reste):
            nb_affixes = len(prefs) + len(sufs)
            # au moins un affixe, une base valide, et plus d'affixes que le meilleur actuel
            if nb_affixes > meilleur_nb and base_valide(base):
                meilleur_nb = nb_affixes
                meilleure = (prefs, base, sufs)
    return meilleure                 # None si aucune décomposition trouvée


for mot in ("défaire", "relire", "chanteur", "chanteuse", "changement", "réutilisable",
            "désorganisation", "surpopulation", "impossible", "lecteur"):
    seg = segmenter(mot)
    if seg is None:
        print(f"{mot:<16} → non analysé")
    else:
        prefs, base, sufs = seg
        print(f"{mot:<16} → {'-'.join(prefs + [base] + sufs)}")

défaire          → dé-faire
relire           → re-lire
chanteur         → chant-eur
chanteuse        → chant-euse
changement       → chang-ement
réutilisable     → ré-utilis-able
désorganisation  → dés-organis-ation
surpopulation    → sur-population
impossible       → im-possible
lecteur          → non analysé


In [35]:
# ✅ Vérification (ne pas modifier)
verifier("réutilisable = ré-utilis-able", lambda: segmenter("réutilisable") == (["ré"], "utilis", ["able"]))
verifier("désorganisation = dés-organis-ation", lambda: segmenter("désorganisation") == (["dés"], "organis", ["ation"]))
verifier("lecteur n'est pas analysé", lambda: segmenter("lecteur") is None)

✅ réutilisable = ré-utilis-able
✅ désorganisation = dés-organis-ation
✅ lecteur n'est pas analysé


#### 📝 Interprétation 1.3
Pourquoi « lecteur » n'est-il pas analysé alors qu'il dérive de « lire » ? Quelle ressource faudrait-il ajouter ? Citez un autre mot dans le même cas.

**Votre réponse :**

« Lecteur » n'est pas analysé parce que le segmenteur retire le suffixe `-eur` et obtient la base `lect`, qui ne peut pas être ramenée à une forme du lexique avec les terminaisons autorisées. La dérivation de « lire » comporte une alternance de base (`lir-` / `lect-`) que les règles simples du programme ne modélisent pas. Il faudrait ajouter un lexique morphologique de formes dérivées et de bases irrégulières, ou des règles explicites d'alternance. « Lecture », dérivé de « lire » avec la base `lect-`, présente le même type de difficulté.

---
## Exercice 2 — Syntaxe : catégories, dépendances et ambiguïté structurelle

La **syntaxe** décrit comment les mots se combinent en syntagmes et en phrases. On utilise deux outils : l'**étiquetage morpho-syntaxique** (catégorie de chaque mot) et l'**analyse en dépendances** (relations tête → dépendant).

### 2.1 Ambiguïté catégorielle
Pour chacune des phrases, affichez la catégorie (`pos_`) attribuée au mot dont le texte commence par « ferm » (utilisez `t.lower_.startswith("ferm")`). Stockez le résultat dans le dictionnaire `categories` : phrase → liste de `(mot, pos)`.

In [36]:
phrases_ferme = ["Le président ferme la porte.", "La ferme est fermée.",
                 "Ils ferment les portes.", "Il parle d'un ton ferme."]
categories = {}
for p in phrases_ferme:
    categories[p] = [(t.text, t.pos_) for t in nlp(p) if t.lower_.startswith("ferm")]
for p, c in categories.items():
    print(f"{p:<32} {c}")

Le président ferme la porte.     [('ferme', 'ADJ')]
La ferme est fermée.             [('ferme', 'ADJ'), ('fermée', 'VERB')]
Ils ferment les portes.          [('ferment', 'VERB')]
Il parle d'un ton ferme.         [('ferme', 'ADJ')]


In [37]:
# ✅ Vérification (ne pas modifier)
verifier("4 phrases analysées", lambda: len(categories) == 4)
verifier("mots en « ferm… » et catégories correctement extraits",
         lambda: all(categories[p] == [(t.text, t.pos_) for t in nlp(p) if t.lower_.startswith("ferm")] for p in phrases_ferme))

✅ 4 phrases analysées
✅ mots en « ferm… » et catégories correctement extraits


#### 📝 Interprétation 2.1
Indiquez, pour chaque phrase, la catégorie attendue de « ferme » (nom, verbe, adjectif). L'étiqueteur se trompe-t-il ? Quels indices du contexte permettent de lever l'ambiguïté ?

**Votre réponse :**

Les catégories attendues sont : « ferme » est un verbe dans « Le président ferme la porte » ; « ferme » est un nom et « fermée » un adjectif (participe employé adjectivement) dans « La ferme est fermée » ; « ferment » est un verbe dans « Ils ferment les portes » ; et « ferme » est un adjectif dans « un ton ferme ». Le modèle se trompe dans les deux premiers contextes : il étiquette le premier « ferme » comme `ADJ`, puis « ferme » comme `ADJ` et « fermée » comme `VERB`. Il réussit sur « ferment » et sur l'adjectif final. Les indices sont la place dans la phrase et les mots voisins : le sujet « président » et le complément « la porte » favorisent le verbe ; l'article « la » indique un nom ; « est » suivi de l'accord féminin favorise un participe/adjectif ; et « ton » introduit un nom modifié par l'adjectif « ferme ».

### 2.2 Dépendances : qui fait quoi ?
Affichez l'arbre de dépendances d'une phrase simple, puis écrivez `svo(phrase)` qui renvoie, pour chaque verbe (`pos_ == "VERB"`), un triplet `(sujets, lemme_du_verbe, objets)` :
- sujets : texte des enfants de relation `nsubj` ou `nsubj:pass` ;
- objets : texte **complet** du sous-arbre des enfants de relation `obj` (`" ".join(x.text for x in c.subtree)`).

In [38]:
doc = nlp("Le chat mange la souris.")
for t in doc:
    print(f"{t.text:<8} --{t.dep_:>6}--> {t.head.text}")
from spacy import displacy
displacy.render(doc, style="dep", jupyter=True, options={"distance": 95})

Le       --   det--> chat
chat     -- nsubj--> mange
mange    --  ROOT--> mange
la       --   det--> souris
souris   --   obj--> mange
.        -- punct--> mange


In [39]:
from spacy.tokens import Doc

def svo(phrase):
    doc_svo = phrase if isinstance(phrase, Doc) else nlp(phrase)     # accepte un texte ou un Doc déjà analysé
    triplets = []
    for v in doc_svo:
        if v.pos_ == "VERB":
            sujets = [c.text for c in v.children if c.dep_ in ("nsubj", "nsubj:pass")]
            objets = [" ".join(x.text for x in c.subtree) for c in v.children if c.dep_ == "obj"]
            triplets.append((sujets, v.lemma_, objets))
    return triplets

for p in ["Le chat mange la souris.",
          "Le professeur explique le cours.",
          "La souris est mangée par le chat."]:
    print(p, "→", svo(p))

Le chat mange la souris. → [(['chat'], 'manger', ['la souris'])]
Le professeur explique le cours. → [(['professeur'], 'expliquer', [])]
La souris est mangée par le chat. → [(['souris'], 'manger', [])]


**Vérification.** Pour tester **votre fonction** indépendamment des erreurs possibles du modèle statistique, la cellule suivante construit à la main des phrases déjà analysées (objets `Doc` dont on fixe les catégories, les relations et les têtes), puis les passe à `svo`.

In [40]:
# ✅ Vérification (ne pas modifier)
def doc_manuel(mots, pos, deps, heads, lemmes):
    """Doc spaCy analysé « à la main » (heads = indice de la tête de chaque mot)."""
    return Doc(nlp.vocab, words=mots, pos=pos, deps=deps, heads=heads, lemmas=lemmes)

d_actif = doc_manuel(["Le", "chat", "mange", "la", "souris", "."],
                     ["DET", "NOUN", "VERB", "DET", "NOUN", "PUNCT"],
                     ["det", "nsubj", "ROOT", "det", "obj", "punct"],
                     [1, 2, 2, 4, 2, 2], ["le", "chat", "manger", "le", "souris", "."])
d_passif = doc_manuel(["La", "souris", "est", "mangée", "par", "le", "chat", "."],
                      ["DET", "NOUN", "AUX", "VERB", "ADP", "DET", "NOUN", "PUNCT"],
                      ["det", "nsubj:pass", "aux:pass", "ROOT", "case", "det", "obl:agent", "punct"],
                      [1, 3, 3, 3, 6, 6, 3, 3], ["le", "souris", "être", "manger", "par", "le", "chat", "."])
verifier("phrase active : (['chat'], 'manger', ['la souris'])", lambda: svo(d_actif) == [(["chat"], "manger", ["la souris"])])
verifier("phrase passive : (['souris'], 'manger', [])", lambda: svo(d_passif) == [(["souris"], "manger", [])])

✅ phrase active : (['chat'], 'manger', ['la souris'])
✅ phrase passive : (['souris'], 'manger', [])


**Quand l'analyseur se trompe.** Exécutez la cellule suivante : votre fonction est correcte, mais le petit modèle `fr_core_news_sm` commet des erreurs d'analyse sur certaines phrases. Observez les catégories et les relations.

In [41]:
for p in ["Le chat noir mange la souris grise.", "Marie lit un roman policier."]:
    print(p, "→", svo(p))
    print("   ", [(t.text, t.pos_, t.dep_, t.head.text) for t in nlp(p)])

Le chat noir mange la souris grise. → [(['chat'], 'manger', []), ([], 'gris', ['la'])]
    [('Le', 'DET', 'det', 'chat'), ('chat', 'NOUN', 'nsubj', 'mange'), ('noir', 'ADJ', 'amod', 'chat'), ('mange', 'VERB', 'ROOT', 'mange'), ('la', 'DET', 'obj', 'grise'), ('souris', 'PROPN', 'dep', 'grise'), ('grise', 'VERB', 'xcomp', 'mange'), ('.', 'PUNCT', 'punct', 'mange')]
Marie lit un roman policier. → [([], 'lire', [])]
    [('Marie', 'NOUN', 'ROOT', 'Marie'), ('lit', 'VERB', 'fixed', 'Marie'), ('un', 'DET', 'det', 'roman'), ('roman', 'NOUN', 'conj', 'Marie'), ('policier', 'ADJ', 'amod', 'roman'), ('.', 'PUNCT', 'punct', 'Marie')]


#### 📝 Interprétation 2.2
Pour « Le chat noir mange la souris grise », quelle catégorie le modèle attribue-t-il à « grise » et quelles relations en découlent ? Pourquoi l'objet de « manger » disparaît-il ? Même question pour « Marie lit un roman policier » (catégorie de « lit » ?). S'agit-il d'une erreur de votre fonction ou de l'analyseur ? Comment la réduire ?

**Votre réponse :**

Dans « Le chat noir mange la souris grise », le modèle étiquette « grise » comme `VERB` et lui donne la relation `xcomp` avec « mange ». Il attache « la » comme `obj` de « grise » et « souris » comme `dep` de « grise » ; « mange » n'a donc aucun enfant `obj`, ce qui explique la disparition de son objet dans le résultat. Dans « Marie lit un roman policier », « lit » est bien `VERB`, mais il reçoit la relation erronée `fixed` et dépend de « Marie », que le modèle analyse comme `NOUN` à la racine. « Marie » n'est donc pas reconnue comme sujet et « roman » est attaché comme `conj`, pas comme objet. Ces résultats viennent des erreurs d'étiquetage et d'analyse en dépendances du modèle, pas de la fonction qui applique les relations demandées. On peut les réduire en utilisant un modèle français plus performant ou spécialisé, en l'adaptant avec des exemples annotés du domaine, puis en contrôlant ou corrigeant les analyses ambiguës.

#### 📝 Interprétation 2.3
Pour la phrase passive « La souris est mangée par le chat », que renvoie votre fonction ? Le sujet grammatical est-il l'agent de l'action ? Comment corriger la fonction (indice : relation `obl:agent`) ?

**Votre réponse :**

La fonction renvoie `[(['souris'], 'manger', [])]` : « souris » est le sujet grammatical passif (`nsubj:pass`) et aucun enfant `obj` n'est attaché au verbe. Ce sujet grammatical désigne le patient, pas l'agent ; l'agent est « le chat », introduit par « par » et relié au verbe par `obl:agent`. Pour obtenir un triplet sémantique actif équivalent, il faut, dans une construction passive, placer l'agent `obl:agent` dans les sujets et le sujet passif `nsubj:pass` dans les objets. On obtiendrait alors `(['chat'], 'manger', ['souris'])`.

---
## ✔️ Avant de rendre
- [ ] Identification complète et nom du fichier conforme
- [ ] `Noyau → Redémarrer et tout exécuter` fonctionne sans erreur
- [ ] Toutes les vérifications ✅ passent
- [ ] Toutes les interprétations 📝 sont rédigées (phrases complètes, appuyées sur vos résultats)

**Bilan personnel (facultatif) :**

J'ai appris à distinguer les informations morphologiques des relations syntaxiques, et à voir qu'une sortie d'analyseur statistique doit être vérifiée dans son contexte. La principale difficulté a été de séparer une erreur de code d'une erreur du modèle spaCy, surtout lorsque celui-ci attribue une mauvaise catégorie ou relation à un mot.